<a href="https://colab.research.google.com/github/Tasnym001/AbbaBashir93/blob/main/GUI_FOR_SOLAR_RADIATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================
# Install dependencies
# ==============================
!pip install ipywidgets scikit-learn pandas openpyxl

# ==============================
# Import libraries
# ==============================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# ==============================
# Load dataset
# ==============================
file_path = "/content/P1.xlsx"  # <-- Change to your Excel file path
data = pd.read_excel(file_path)

# Ensure correct columns exist
expected_cols = ['DUST', 'RAIN', 'WD', 'WS', 'TEMP', 'RH', 'BP', 'SOLAR']
if not all(col in data.columns for col in expected_cols):
    raise ValueError(f"Dataset must include columns: {expected_cols}")

X = data[['DUST', 'RAIN', 'WD', 'WS', 'TEMP', 'RH', 'BP']]
y = data['SOLAR']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==============================
# Split data
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ==============================
# Train Random Forest with 5-fold CV
# ==============================
print("Training Random Forest Regressor with 5-fold cross-validation...")

rf_model = RandomForestRegressor(
    n_estimators=1000,
    max_depth=12,
    min_samples_split=6,
    min_samples_leaf=3,
    random_state=42
)

# Perform cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X_scaled, y, cv=cv, scoring='r2')

# Fit model on training data
rf_model.fit(X_train, y_train)

# Evaluate performance
y_pred = rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n===== Random Forest Model Performance =====")
print(f"Cross-Validation R² Scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f}")
print(f"Test R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print("==========================================")

# ==============================
# GUI INTERFACE (UNCHANGED STYLE)
# ==============================

title_authors_box = widgets.HTML(
    value="""
    <div style="
        background-color:#4CAF50;
        color:white;
        padding:15px;
        border-radius:10px;
        text-align:center;">
        <h2 style="margin:5px;color:black;">GUI for Solar Radiation Prediction in Riyadh</h2>
        <p style="margin:5px; font-size:16px; color:black;"><b>Developed by:</b> Abba Bashir, Naif K. Alshammari, Sani I. Abba</p>
    </div>
    """
)

# --- Input Features ---
feature_cols = ['DUST', 'RAIN', 'WD', 'WS', 'TEMP', 'RH', 'BP']

input_widgets = {
    "DUST": widgets.FloatText(description="DUST"),
    "RAIN": widgets.FloatText(description="RAIN"),
    "WD": widgets.FloatText(description="WD"),
    "WS": widgets.FloatText(description="WS"),
    "TEMP": widgets.FloatText(description="TEMP"),
    "RH": widgets.FloatText(description="RH"),
    "BP": widgets.FloatText(description="BP"),
}

input_label = widgets.HTML(
    "<div style='background-color:#4CAF50; text-align:center; font-weight:bold; font-size:16px;'>Input Meteorological Features</div>"
)
inputs_box = widgets.VBox(list(input_widgets.values()))

inputs_container = widgets.VBox(
    [input_label, inputs_box],
    layout=widgets.Layout(
        border="5px solid #4CAF50",
        padding="15px",
        margin="15px",
        background_color="#90EE90",
        border_radius="12px",
        width="100%"
    )
)

# --- Output Section ---
output_label = widgets.HTML(
    "<div style='background-color:#4CAF50; text-align:center; font-weight:bold; font-size:16px;'>Prediction Result</div>"
)

solar_out = widgets.HTML("<b>Predicted Solar Radiation (SOLAR):</b> - W/m²")

output_box = widgets.VBox([solar_out])

outputs_container = widgets.VBox(
    [output_label, output_box],
    layout=widgets.Layout(
        border="5px solid #4CAF50",
        padding="15px",
        margin="15px",
        background_color="#90EE90",
        border_radius="12px",
        width="100%"
    )
)

# --- Buttons ---
predict_button = widgets.Button(
    description="Predict",
    button_style="success",
    layout=widgets.Layout(width="150px")
)

reset_button = widgets.Button(
    description="Reset",
    button_style="danger",
    layout=widgets.Layout(width="150px")
)

buttons_box = widgets.HBox(
    [predict_button, reset_button],
    layout=widgets.Layout(justify_content="center")
)

# --- Button Logic ---
def on_predict_clicked(b):
    try:
        input_values = [input_widgets[feature].value for feature in feature_cols]
        input_array = np.array(input_values).reshape(1, -1)
        input_scaled = scaler.transform(input_array)

        solar_pred = rf_model.predict(input_scaled)[0]
        solar_out.value = f"<b>Predicted Solar Radiation (SOLAR):</b> {solar_pred:.2f} W/m²"

    except Exception as e:
        solar_out.value = f"<b>Error:</b> {str(e)}"

def on_reset_clicked(b):
    for widget in input_widgets.values():
        widget.value = 0.0
    solar_out.value = "<b>Predicted Solar Radiation (SOLAR):</b> - W/m²"

predict_button.on_click(on_predict_clicked)
reset_button.on_click(on_reset_clicked)

# --- Layout ---
grid_layout = widgets.HBox([inputs_container, outputs_container])

app_layout = widgets.VBox([
    title_authors_box,
    widgets.HTML("<hr style='border:10px solid #4CAF50; margin:20px 0;'>"),
    grid_layout,
    buttons_box
])

display(app_layout)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.2 MB/s eta 0:00:00
Training Random Forest Regressor with 5-fold cross-validation...

===== Random Forest Model Performance =====
Cross-Validation R² Scores: [0.96792664 0.9708244  0.93388742 0.97666072 0.97160146]
Mean CV R²: 0.9642
Test R²: 0.9680
MAE: 176.3430
RMSE: 398.4559
